In [0]:
# ADLS access permission
storage_account = "storage_account_name"
access_key = "storage_account_key"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)

In [0]:
# Read Data from OpenF1 API
import requests
url = "https://api.openf1.org/v1/drivers?session_key=latest"
data = requests.get(url).json()
print(f"Records fetched: {len(data)}")

Records fetched: 22


In [0]:
# Convert JSON to Spark Dataframe
import pandas as pd
pdf = pd.DataFrame(data)
pdf = pdf.fillna("")
df = spark.createDataFrame(pdf)
display(df)

df.printSchema()

meeting_key,session_key,driver_number,broadcast_name,full_name,name_acronym,team_name,team_colour,first_name,last_name,headshot_url,country_code
1287,11307,1,L NORRIS,Lando NORRIS,NOR,McLaren,F47600,Lando,Norris,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/L/LANNOR01_Lando_Norris/lannor01.png.transform/1col/image.png,
1287,11307,3,M VERSTAPPEN,Max VERSTAPPEN,VER,Red Bull Racing,4781D7,Max,Verstappen,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/M/MAXVER01_Max_Verstappen/maxver01.png.transform/1col/image.png,
1287,11307,5,G BORTOLETO,Gabriel BORTOLETO,BOR,Audi,F50537,Gabriel,Bortoleto,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/G/GABBOR01_Gabriel_Bortoleto/gabbor01.png.transform/1col/image.png,
1287,11307,6,I HADJAR,Isack HADJAR,HAD,Red Bull Racing,4781D7,Isack,Hadjar,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/I/ISAHAD01_Isack_Hadjar/isahad01.png.transform/1col/image.png,
1287,11307,10,P GASLY,Pierre GASLY,GAS,Alpine,00A1E8,Pierre,Gasly,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/P/PIEGAS01_Pierre_Gasly/piegas01.png.transform/1col/image.png,
1287,11307,11,S PEREZ,Sergio PEREZ,PER,Cadillac,909090,Sergio,Perez,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/S/SERPER01_Sergio_Perez/serper01.png.transform/1col/image.png,
1287,11307,12,K ANTONELLI,Kimi ANTONELLI,ANT,Mercedes,00D7B6,Kimi,Antonelli,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/K/ANDANT01_Kimi_Antonelli/andant01.png.transform/1col/image.png,
1287,11307,14,F ALONSO,Fernando ALONSO,ALO,Aston Martin,229971,Fernando,Alonso,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/F/FERALO01_Fernando_Alonso/feralo01.png.transform/1col/image.png,
1287,11307,16,C LECLERC,Charles LECLERC,LEC,Ferrari,ED1131,Charles,Leclerc,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/C/CHALEC01_Charles_Leclerc/chalec01.png.transform/1col/image.png,
1287,11307,18,L STROLL,Lance STROLL,STR,Aston Martin,229971,Lance,Stroll,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/L/LANSTR01_Lance_Stroll/lanstr01.png.transform/1col/image.png,


root
 |-- meeting_key: long (nullable = true)
 |-- session_key: long (nullable = true)
 |-- driver_number: long (nullable = true)
 |-- broadcast_name: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- name_acronym: string (nullable = true)
 |-- team_name: string (nullable = true)
 |-- team_colour: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- headshot_url: string (nullable = true)
 |-- country_code: string (nullable = true)



In [0]:
# save to raw layer
df.write \
.mode("overwrite") \
.json(
    "abfss://raww@driversdatastorage.dfs.core.windows.net/drivers_api"
)

In [0]:
# container access verifying
dbutils.fs.ls(
    "abfss://raww@driversdatastorage.dfs.core.windows.net/"
)

[FileInfo(path='abfss://raww@driversdatastorage.dfs.core.windows.net/drivers_api/', name='drivers_api/', size=0, modificationTime=1781248474000)]

In [0]:
# save to delta(to processed layer)
df.write \
.format("delta") \
.mode("overwrite") \
.save("abfss://processed@driversdatastorage.dfs.core.windows.net/drivers")

In [0]:
# read from delta
processed_df = spark.read.format("delta") \
.load("abfss://processed@driversdatastorage.dfs.core.windows.net/drivers")

display(processed_df)


# container check
dbutils.fs.ls(
    "abfss://processed@driversdatastorage.dfs.core.windows.net/"
)

meeting_key,session_key,driver_number,broadcast_name,full_name,name_acronym,team_name,team_colour,first_name,last_name,headshot_url,country_code
1287,11307,1,L NORRIS,Lando NORRIS,NOR,McLaren,F47600,Lando,Norris,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/L/LANNOR01_Lando_Norris/lannor01.png.transform/1col/image.png,
1287,11307,3,M VERSTAPPEN,Max VERSTAPPEN,VER,Red Bull Racing,4781D7,Max,Verstappen,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/M/MAXVER01_Max_Verstappen/maxver01.png.transform/1col/image.png,
1287,11307,5,G BORTOLETO,Gabriel BORTOLETO,BOR,Audi,F50537,Gabriel,Bortoleto,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/G/GABBOR01_Gabriel_Bortoleto/gabbor01.png.transform/1col/image.png,
1287,11307,6,I HADJAR,Isack HADJAR,HAD,Red Bull Racing,4781D7,Isack,Hadjar,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/I/ISAHAD01_Isack_Hadjar/isahad01.png.transform/1col/image.png,
1287,11307,10,P GASLY,Pierre GASLY,GAS,Alpine,00A1E8,Pierre,Gasly,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/P/PIEGAS01_Pierre_Gasly/piegas01.png.transform/1col/image.png,
1287,11307,11,S PEREZ,Sergio PEREZ,PER,Cadillac,909090,Sergio,Perez,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/S/SERPER01_Sergio_Perez/serper01.png.transform/1col/image.png,
1287,11307,12,K ANTONELLI,Kimi ANTONELLI,ANT,Mercedes,00D7B6,Kimi,Antonelli,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/K/ANDANT01_Kimi_Antonelli/andant01.png.transform/1col/image.png,
1287,11307,14,F ALONSO,Fernando ALONSO,ALO,Aston Martin,229971,Fernando,Alonso,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/F/FERALO01_Fernando_Alonso/feralo01.png.transform/1col/image.png,
1287,11307,16,C LECLERC,Charles LECLERC,LEC,Ferrari,ED1131,Charles,Leclerc,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/C/CHALEC01_Charles_Leclerc/chalec01.png.transform/1col/image.png,
1287,11307,18,L STROLL,Lance STROLL,STR,Aston Martin,229971,Lance,Stroll,https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/L/LANSTR01_Lance_Stroll/lanstr01.png.transform/1col/image.png,


[FileInfo(path='abfss://processed@driversdatastorage.dfs.core.windows.net/drivers/', name='drivers/', size=0, modificationTime=1781248699000)]